# AHP-Gaussian for Project Portfolio Prioritization

This notebook applies the AHP-Gaussian method to project portfolio prioritization by incorporating the variability of the evaluation criteria into the weighting process.

## 1. Objective

The objective of this analysis is to apply the AHP-Gaussian method to prioritize projects based on multiple evaluation criteria.

Unlike the conventional AHP approach, the AHP-Gaussian method incorporates the coefficient of variation of each criterion into the weighting process, allowing the variability of the data to influence the relative importance of the criteria.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

In [4]:
df = pd.read_csv("../data/raw/projects.csv")

df.shape

(4000, 51)

## 2. Define Evaluation Criteria

The projects are evaluated using six criteria. Each criterion is classified as either a maximization or minimization criterion according to its contribution to project prioritization.

In [5]:
criteria_objectives = {
    "Project_Budget_USD": "min",
    "Estimated_Timeline_Months": "min",
    "Complexity_Score": "min",
    "Previous_Delivery_Success_Rate": "max",
    "Resource_Availability": "max",
    "Historical_Risk_Incidents": "min",
}

criteria_names = list(criteria_objectives.keys())

criteria_names

['Project_Budget_USD',
 'Estimated_Timeline_Months',
 'Complexity_Score',
 'Previous_Delivery_Success_Rate',
 'Resource_Availability',
 'Historical_Risk_Incidents']

## 3. Calculate Criterion Statistics

The mean and standard deviation of each criterion are calculated to measure the variability of the project portfolio.

In [6]:
criterion_stats = df[criteria_names].agg(
    ["mean", "std"]
).T

criterion_stats

,mean,std
Project_Budget_USD,1.143032e+06,590878.115350
Estimated_Timeline_Months,1.714775e+01,6.926609
Complexity_Score,6.192525e+00,2.212538
Previous_Delivery_Success_Rate,7.504375e-01,0.143712
Resource_Availability,6.516950e-01,0.201163
Historical_Risk_Incidents,1.507250e+00,1.210915


## 4. Calculate the Coefficient of Variation

The coefficient of variation (CV) is calculated for each criterion as the ratio between its standard deviation and mean.

The CV represents the relative variability of each criterion and is used by the AHP-Gaussian method to incorporate data dispersion into the criterion weighting process.

In [7]:
criterion_stats["CV"] = (
    criterion_stats["std"]
    / criterion_stats["mean"]
)

criterion_stats

,mean,std,CV
Project_Budget_USD,1.143032e+06,590878.115350,0.516939
Estimated_Timeline_Months,1.714775e+01,6.926609,0.403937
Complexity_Score,6.192525e+00,2.212538,0.357292
Previous_Delivery_Success_Rate,7.504375e-01,0.143712,0.191504
Resource_Availability,6.516950e-01,0.201163,0.308677
Historical_Risk_Incidents,1.507250e+00,1.210915,0.803393


In [8]:
criterion_stats[
    ["mean", "std", "CV"]
]

,mean,std,CV
Project_Budget_USD,1.143032e+06,590878.115350,0.516939
Estimated_Timeline_Months,1.714775e+01,6.926609,0.403937
Complexity_Score,6.192525e+00,2.212538,0.357292
Previous_Delivery_Success_Rate,7.504375e-01,0.143712,0.191504
Resource_Availability,6.516950e-01,0.201163,0.308677
Historical_Risk_Incidents,1.507250e+00,1.210915,0.803393


## 5. Calculate the Gaussian Factor

The coefficient of variation is transformed into a Gaussian factor to represent the relative importance associated with the variability of each criterion.

The Gaussian factor is calculated using the exponential function:

\[
G_i = e^{-\frac{CV_i^2}{2}}
\]

where \(CV_i\) is the coefficient of variation of criterion \(i\).

In [9]:
criterion_stats["Gaussian_Factor"] = np.exp(
    -(criterion_stats["CV"] ** 2) / 2
)

criterion_stats[
    ["mean", "std", "CV", "Gaussian_Factor"]
]

,mean,std,CV,Gaussian_Factor
Project_Budget_USD,1.143032e+06,590878.115350,0.516939,0.874928
Estimated_Timeline_Months,1.714775e+01,6.926609,0.403937,0.921657
Complexity_Score,6.192525e+00,2.212538,0.357292,0.938166
Previous_Delivery_Success_Rate,7.504375e-01,0.143712,0.191504,0.981830
Resource_Availability,6.516950e-01,0.201163,0.308677,0.953476
Historical_Risk_Incidents,1.507250e+00,1.210915,0.803393,0.724176


In [10]:
criterion_stats[
    ["CV", "Gaussian_Factor"]
].sort_values(
    by="Gaussian_Factor",
    ascending=False
)

,CV,Gaussian_Factor
Previous_Delivery_Success_Rate,0.191504,0.981830
Resource_Availability,0.308677,0.953476
Complexity_Score,0.357292,0.938166
Estimated_Timeline_Months,0.403937,0.921657
Project_Budget_USD,0.516939,0.874928
Historical_Risk_Incidents,0.803393,0.724176


## 6. Calculate AHP Criteria Weights

The conventional AHP weights are calculated from the pairwise comparison matrix. These weights represent the relative importance of each evaluation criterion before incorporating the Gaussian factor.

In [11]:
pairwise_matrix = pd.DataFrame(
    [
        [1,   1/3, 1/5, 1/7, 1/3, 1/5],
        [3,   1,   1/3, 1/5, 1/2, 1/3],
        [5,   3,   1,   1/3, 3,   1],
        [7,   5,   3,   1,   5,   3],
        [3,   2,   1/3, 1/5, 1,   1/2],
        [5,   3,   1,   1/3, 2,   1],
    ],
    index=criteria_names,
    columns=criteria_names,
)

normalized_pairwise = pairwise_matrix.div(
    pairwise_matrix.sum(axis=0),
    axis=1
)

ahp_weights = normalized_pairwise.mean(axis=1)

ahp_weights

Project_Budget_USD                0.037498
Estimated_Timeline_Months         0.073268
Complexity_Score                  0.193037
Previous_Delivery_Success_Rate    0.420704
Resource_Availability             0.096542
Historical_Risk_Incidents         0.178952
dtype: float64

In [12]:
ahp_weights.sum()

0.9999999999999999

In [13]:
ahp_weights.sort_values(
    ascending=False
)

Previous_Delivery_Success_Rate    0.420704
Complexity_Score                  0.193037
Historical_Risk_Incidents         0.178952
Resource_Availability             0.096542
Estimated_Timeline_Months         0.073268
Project_Budget_USD                0.037498
dtype: float64

## 7. Calculate Gaussian-Adjusted Weights

The Gaussian-adjusted weights combine the conventional AHP weights with the Gaussian factor calculated from the coefficient of variation.

The resulting weights incorporate both the subjective importance defined through AHP and the relative variability observed in the project portfolio.

In [14]:
gaussian_weights = (
    ahp_weights
    * criterion_stats["Gaussian_Factor"]
)

gaussian_weights = (
    gaussian_weights
    / gaussian_weights.sum()
)

gaussian_weights

Project_Budget_USD                0.035811
Estimated_Timeline_Months         0.073709
Complexity_Score                  0.197678
Previous_Delivery_Success_Rate    0.450871
Resource_Availability             0.100476
Historical_Risk_Incidents         0.141455
dtype: float64

In [15]:
weights_comparison = pd.DataFrame({
    "AHP_Weight": ahp_weights,
    "Gaussian_Factor": criterion_stats["Gaussian_Factor"],
    "AHP_Gaussian_Weight": gaussian_weights,
})

weights_comparison.sort_values(
    by="AHP_Gaussian_Weight",
    ascending=False
)

,AHP_Weight,Gaussian_Factor,AHP_Gaussian_Weight
Previous_Delivery_Success_Rate,0.420704,0.981830,0.450871
Complexity_Score,0.193037,0.938166,0.197678
Historical_Risk_Incidents,0.178952,0.724176,0.141455
Resource_Availability,0.096542,0.953476,0.100476
Estimated_Timeline_Months,0.073268,0.921657,0.073709
Project_Budget_USD,0.037498,0.874928,0.035811


In [16]:
gaussian_weights.sum()

1.0

## 8. Normalize the Decision Matrix

The decision matrix is normalized so that the criteria can be combined using a common scale.

For maximization criteria, higher values receive higher normalized scores. For minimization criteria, lower values receive higher normalized scores.

In [17]:
decision_matrix = df[criteria_names].copy()

normalized_decision = decision_matrix.copy()

for criterion, objective in criteria_objectives.items():
    minimum = decision_matrix[criterion].min()
    maximum = decision_matrix[criterion].max()

    if maximum == minimum:
        normalized_decision[criterion] = 1

    elif objective == "max":
        normalized_decision[criterion] = (
            decision_matrix[criterion] - minimum
        ) / (maximum - minimum)

    elif objective == "min":
        normalized_decision[criterion] = (
            maximum - decision_matrix[criterion]
        ) / (maximum - minimum)

    else:
        raise ValueError(
            f"Invalid objective: {objective}"
        )

normalized_decision.head()

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
0,0.621246,0.117647,0.035800,0.773810,0.971429,0.750
1,0.935873,0.794118,0.868735,0.690476,0.928571,0.750
2,0.975805,0.882353,0.949881,0.904762,0.700000,0.750
3,0.648524,0.558824,0.293556,0.666667,0.314286,0.875
4,0.574012,0.352941,0.396181,0.809524,0.400000,0.875


In [18]:
normalized_decision.describe()

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,0.727438,0.554478,0.454353,0.714807,0.502421,0.811594
std,0.163724,0.203724,0.264026,0.171085,0.287376,0.151364
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.635213,0.411765,0.255072,0.607143,0.257143,0.750000
50%,0.765000,0.558824,0.475537,0.738095,0.500000,0.875000
75%,0.852264,0.705882,0.661098,0.845238,0.742857,0.875000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 9. Calculate AHP-Gaussian Scores

The normalized decision matrix is combined with the Gaussian-adjusted weights to calculate an overall score for each project.

Higher scores indicate projects with more favorable characteristics according to the selected criteria and their adjusted weights.

In [19]:
weighted_gaussian_matrix = normalized_decision.mul(
    gaussian_weights,
    axis=1
)

ahp_gaussian_scores = weighted_gaussian_matrix.sum(
    axis=1
)

ahp_gaussian_scores.head()

0    0.590581
1    0.774484
2    0.872108
3    0.578376
4    0.653842
dtype: float64

In [20]:
ahp_gaussian_scores.describe()

count    4000.000000
mean        0.644307
std         0.111265
min         0.219389
25%         0.570112
50%         0.650367
75%         0.726436
max         0.945826
dtype: float64

In [21]:
print("Minimum score:", ahp_gaussian_scores.min())
print("Maximum score:", ahp_gaussian_scores.max())

Minimum score: 0.21938941746672597
Maximum score: 0.9458259001254826


## 10. Generate the AHP-Gaussian Ranking

The projects are ranked according to their AHP-Gaussian scores, with higher scores receiving higher priority.

In [22]:
ahp_gaussian_ranking = df[["Project_ID"]].copy()

ahp_gaussian_ranking["AHP_Gaussian_Score"] = (
    ahp_gaussian_scores
)

ahp_gaussian_ranking = ahp_gaussian_ranking.sort_values(
    by="AHP_Gaussian_Score",
    ascending=False
).reset_index(drop=True)

ahp_gaussian_ranking["AHP_Gaussian_Rank"] = (
    ahp_gaussian_ranking.index + 1
)

ahp_gaussian_ranking.head(10)

,Project_ID,AHP_Gaussian_Score,AHP_Gaussian_Rank
0,PROJ_0614,0.945826,1
1,PROJ_3497,0.935679,2
2,PROJ_1686,0.932203,3
3,PROJ_0940,0.930098,4
4,PROJ_2031,0.926690,5
5,PROJ_2119,0.921286,6
6,PROJ_3982,0.918222,7
7,PROJ_2845,0.914946,8
8,PROJ_1070,0.911107,9
9,PROJ_2068,0.910133,10


In [23]:
ahp_gaussian_ranking.tail(5)

,Project_ID,AHP_Gaussian_Score,AHP_Gaussian_Rank
3995,PROJ_1631,0.304427,3996
3996,PROJ_2713,0.279611,3997
3997,PROJ_0265,0.248807,3998
3998,PROJ_3899,0.242853,3999
3999,PROJ_1487,0.219389,4000


In [24]:
print("Number of projects:", len(ahp_gaussian_ranking))
print(
    "Number of unique projects:",
    ahp_gaussian_ranking["Project_ID"].nunique()
)
print(
    "Number of unique ranks:",
    ahp_gaussian_ranking["AHP_Gaussian_Rank"].nunique()
)

Number of projects: 4000
Number of unique projects: 4000
Number of unique ranks: 4000


## 11. Compare AHP and AHP-Gaussian Rankings

The AHP and AHP-Gaussian rankings are compared to identify how incorporating criterion variability affects project prioritization.

In [25]:
ahp_ranking = pd.read_csv(
    "../data/processed/ahp_ranking.csv"
)

ahp_ranking.head()

,Project_ID,Project_Type,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Methodology_Used,Team_Experience_Level,Past_Similar_Projects,...,Client_Experience_Level,Change_Control_Maturity,Risk_Management_Maturity,Team_Colocation,Documentation_Quality,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor,Risk_Level,AHP_Score
0,PROJ_0614,Manufacturing,3,265992.55,4,1.91,3,Hybrid,Senior,5,...,First-time,Basic,Basic,Fully Remote,Good,11,1,1.0,High,0.948052
1,PROJ_3497,Manufacturing,9,388789.62,5,2.44,8,Waterfall,Senior,5,...,First-time,Basic,NaN,Fully Remote,Basic,8,1,1.0,Medium,0.937929
2,PROJ_1686,Healthcare,2,428345.18,4,1.89,7,Scrum,Junior,1,...,First-time,Formal,Basic,Hybrid,Excellent,8,1,1.0,Medium,0.935575
3,PROJ_0940,Healthcare,4,491707.71,4,2.04,6,Scrum,Mixed,2,...,Occasional,Formal,NaN,Fully Colocated,Good,12,1,1.0,Low,0.932549
4,PROJ_2031,Manufacturing,2,310468.99,3,1.67,4,Kanban,Junior,1,...,Occasional,Formal,Advanced,Fully Remote,Basic,7,1,1.0,Low,0.930085


In [26]:
ahp_ranking = pd.read_csv(
    "../data/processed/ahp_ranking.csv"
)

ahp_ranking["AHP_Rank"] = (
    ahp_ranking["AHP_Score"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

ahp_comparison = ahp_ranking[
    ["Project_ID", "AHP_Rank", "AHP_Score"]
].merge(
    ahp_gaussian_ranking[
        ["Project_ID", "AHP_Gaussian_Rank", "AHP_Gaussian_Score"]
    ],
    on="Project_ID",
    how="inner"
)

ahp_comparison.head()

,Project_ID,AHP_Rank,AHP_Score,AHP_Gaussian_Rank,AHP_Gaussian_Score
0,PROJ_0614,1,0.948052,1,0.945826
1,PROJ_3497,2,0.937929,2,0.935679
2,PROJ_1686,3,0.935575,3,0.932203
3,PROJ_0940,4,0.932549,4,0.930098
4,PROJ_2031,5,0.930085,5,0.926690


In [27]:
ahp_comparison["Rank_Change"] = (
    ahp_comparison["AHP_Rank"]
    - ahp_comparison["AHP_Gaussian_Rank"]
)

In [28]:
ahp_comparison.sort_values(
    by="Rank_Change",
    ascending=False
).head(10)

,Project_ID,AHP_Rank,AHP_Score,AHP_Gaussian_Rank,AHP_Gaussian_Score,Rank_Change
2352,PROJ_1538,2353,0.628908,1867,0.660063,486
2577,PROJ_0668,2578,0.612561,2126,0.641909,452
2089,PROJ_2247,2090,0.649628,1675,0.674390,415
2150,PROJ_3295,2151,0.645344,1736,0.669841,415
2620,PROJ_2498,2621,0.609571,2268,0.629841,353
2349,PROJ_3989,2350,0.629180,2008,0.649899,342
2078,PROJ_0336,2079,0.650402,1745,0.669339,334
2585,PROJ_0071,2586,0.612172,2278,0.629151,308
1958,PROJ_1592,1959,0.659088,1658,0.675904,301
2266,PROJ_1629,2267,0.636123,1976,0.652333,291


## 12. Compare Ranking Correlation

Spearman's rank correlation is used to measure the similarity between the AHP and AHP-Gaussian project rankings.

A value close to 1 indicates that the two methods produce very similar rankings, while lower values indicate greater differences in project ordering.

In [29]:
from scipy.stats import spearmanr

In [30]:
spearman_correlation, p_value = spearmanr(
    ahp_comparison["AHP_Rank"],
    ahp_comparison["AHP_Gaussian_Rank"]
)

print(
    f"Spearman rank correlation: "
    f"{spearman_correlation:.4f}"
)

print(
    f"P-value: "
    f"{p_value:.4e}"
)

Spearman rank correlation: 0.9977
P-value: 0.0000e+00


In [31]:
print(
    "Projects compared:",
    len(ahp_comparison)
)

Projects compared: 4000


## 13. Compare the Top 20 Projects

The Top 20 projects from the AHP and AHP-Gaussian rankings are compared to identify which projects remain highly prioritized and which projects enter or leave the highest-priority group.

In [32]:
top_20_ahp = set(
    ahp_comparison
    .nsmallest(20, "AHP_Rank")["Project_ID"]
)

top_20_gaussian = set(
    ahp_comparison
    .nsmallest(20, "AHP_Gaussian_Rank")["Project_ID"]
)

top_20_overlap = (
    top_20_ahp
    & top_20_gaussian
)

print(
    f"Top-20 overlap: "
    f"{len(top_20_overlap)}/20 "
    f"({len(top_20_overlap) / 20 * 100:.1f}%)"
)

Top-20 overlap: 18/20 (90.0%)


In [33]:
top_20_only_ahp = sorted(
    top_20_ahp - top_20_gaussian
)

top_20_only_gaussian = sorted(
    top_20_gaussian - top_20_ahp
)

print("Projects only in AHP Top 20:")
print(top_20_only_ahp)

print("\nProjects only in AHP-Gaussian Top 20:")
print(top_20_only_gaussian)

Projects only in AHP Top 20:
['PROJ_2342', 'PROJ_3499']

Projects only in AHP-Gaussian Top 20:
['PROJ_0692', 'PROJ_2838']


In [34]:
top_20_comparison = ahp_comparison[
    ahp_comparison["Project_ID"].isin(
        top_20_ahp | top_20_gaussian
    )
].sort_values(
    by="AHP_Gaussian_Rank"
)

top_20_comparison[
    [
        "Project_ID",
        "AHP_Rank",
        "AHP_Gaussian_Rank",
        "Rank_Change",
        "AHP_Score",
        "AHP_Gaussian_Score",
    ]
]

,Project_ID,AHP_Rank,AHP_Gaussian_Rank,Rank_Change,AHP_Score,AHP_Gaussian_Score
0,PROJ_0614,1,1,0,0.948052,0.945826
1,PROJ_3497,2,2,0,0.937929,0.935679
2,PROJ_1686,3,3,0,0.935575,0.932203
3,PROJ_0940,4,4,0,0.932549,0.930098
4,PROJ_2031,5,5,0,0.930085,0.926690
8,PROJ_2119,9,6,3,0.913478,0.921286
5,PROJ_3982,6,7,-1,0.920823,0.918222
6,PROJ_2845,7,8,-1,0.918382,0.914946
7,PROJ_1070,8,9,-1,0.914523,0.911107
9,PROJ_2068,10,10,0,0.913064,0.910133


## 14. Analyze the Largest Rank Changes

The absolute change in ranking position is used to identify the projects most affected by the incorporation of criterion variability.

Projects with larger absolute changes experienced a greater impact from the AHP-Gaussian adjustment.

In [35]:
ahp_comparison["Absolute_Rank_Change"] = (
    ahp_comparison["Rank_Change"].abs()
)

In [36]:
largest_improvements = ahp_comparison.sort_values(
    by="Rank_Change",
    ascending=False
).head(10)

largest_improvements[
    [
        "Project_ID",
        "AHP_Rank",
        "AHP_Gaussian_Rank",
        "Rank_Change",
        "AHP_Score",
        "AHP_Gaussian_Score",
    ]
]

,Project_ID,AHP_Rank,AHP_Gaussian_Rank,Rank_Change,AHP_Score,AHP_Gaussian_Score
2352,PROJ_1538,2353,1867,486,0.628908,0.660063
2577,PROJ_0668,2578,2126,452,0.612561,0.641909
2089,PROJ_2247,2090,1675,415,0.649628,0.674390
2150,PROJ_3295,2151,1736,415,0.645344,0.669841
2620,PROJ_2498,2621,2268,353,0.609571,0.629841
2349,PROJ_3989,2350,2008,342,0.629180,0.649899
2078,PROJ_0336,2079,1745,334,0.650402,0.669339
2585,PROJ_0071,2586,2278,308,0.612172,0.629151
1958,PROJ_1592,1959,1658,301,0.659088,0.675904
2266,PROJ_1629,2267,1976,291,0.636123,0.652333


In [37]:
largest_declines = ahp_comparison.sort_values(
    by="Rank_Change",
    ascending=True
).head(10)

largest_declines[
    [
        "Project_ID",
        "AHP_Rank",
        "AHP_Gaussian_Rank",
        "Rank_Change",
        "AHP_Score",
        "AHP_Gaussian_Score",
    ]
]

,Project_ID,AHP_Rank,AHP_Gaussian_Rank,Rank_Change,AHP_Score,AHP_Gaussian_Score
1892,PROJ_2042,1893,2113,-220,0.662555,0.642758
2458,PROJ_0012,2459,2678,-219,0.619955,0.597337
1694,PROJ_2536,1695,1911,-216,0.677949,0.657034
1836,PROJ_3876,1837,2045,-208,0.666445,0.647336
2545,PROJ_1490,2546,2747,-201,0.613880,0.592130
1858,PROJ_3675,1859,2059,-200,0.664954,0.646103
1785,PROJ_0092,1786,1977,-191,0.669892,0.652269
2035,PROJ_1961,2036,2225,-189,0.653291,0.633691
2634,PROJ_0238,2635,2821,-186,0.608175,0.586932
2651,PROJ_1043,2652,2837,-185,0.607004,0.585348


In [38]:
print(
    "Mean absolute rank change:",
    ahp_comparison["Absolute_Rank_Change"].mean()
)

print(
    "Maximum absolute rank change:",
    ahp_comparison["Absolute_Rank_Change"].max()
)

Mean absolute rank change: 58.3965
Maximum absolute rank change: 486


## 15. Analyze Criterion Weight Changes

The AHP and AHP-Gaussian weights are compared to identify how the incorporation of criterion variability changes the relative importance of each evaluation criterion.

In [39]:
weights_comparison["Weight_Change"] = (
    weights_comparison["AHP_Gaussian_Weight"]
    - weights_comparison["AHP_Weight"]
)

weights_comparison["Absolute_Weight_Change"] = (
    weights_comparison["Weight_Change"].abs()
)

weights_comparison.sort_values(
    by="AHP_Gaussian_Weight",
    ascending=False
)

,AHP_Weight,Gaussian_Factor,AHP_Gaussian_Weight,Weight_Change,Absolute_Weight_Change
Previous_Delivery_Success_Rate,0.420704,0.981830,0.450871,0.030166,0.030166
Complexity_Score,0.193037,0.938166,0.197678,0.004641,0.004641
Historical_Risk_Incidents,0.178952,0.724176,0.141455,-0.037497,0.037497
Resource_Availability,0.096542,0.953476,0.100476,0.003935,0.003935
Estimated_Timeline_Months,0.073268,0.921657,0.073709,0.000441,0.000441
Project_Budget_USD,0.037498,0.874928,0.035811,-0.001687,0.001687


In [40]:
print("Criteria with increased weight:")
print(
    weights_comparison[
        weights_comparison["Weight_Change"] > 0
    ].sort_values(
        by="Weight_Change",
        ascending=False
    )
)

print("\nCriteria with decreased weight:")
print(
    weights_comparison[
        weights_comparison["Weight_Change"] < 0
    ].sort_values(
        by="Weight_Change"
    )
)

Criteria with increased weight:
                                AHP_Weight  Gaussian_Factor  \
Previous_Delivery_Success_Rate    0.420704         0.981830   
Complexity_Score                  0.193037         0.938166   
Resource_Availability             0.096542         0.953476   
Estimated_Timeline_Months         0.073268         0.921657   

                                AHP_Gaussian_Weight  Weight_Change  \
Previous_Delivery_Success_Rate             0.450871       0.030166   
Complexity_Score                           0.197678       0.004641   
Resource_Availability                      0.100476       0.003935   
Estimated_Timeline_Months                  0.073709       0.000441   

                                Absolute_Weight_Change  
Previous_Delivery_Success_Rate                0.030166  
Complexity_Score                              0.004641  
Resource_Availability                         0.003935  
Estimated_Timeline_Months                     0.000441  

Criteria with d

In [41]:
print(
    "AHP weights sum:",
    weights_comparison["AHP_Weight"].sum()
)

print(
    "AHP-Gaussian weights sum:",
    weights_comparison["AHP_Gaussian_Weight"].sum()
)

AHP weights sum: 0.9999999999999999
AHP-Gaussian weights sum: 1.0


## 16. Identify Projects with the Largest Ranking Changes

The projects with the largest changes in ranking position are identified to evaluate which projects are most affected by the incorporation of criterion variability.

In [42]:
largest_rank_changes = ahp_comparison.sort_values(
    by="Absolute_Rank_Change",
    ascending=False
).head(20)

largest_rank_changes[
    [
        "Project_ID",
        "AHP_Rank",
        "AHP_Gaussian_Rank",
        "Rank_Change",
        "Absolute_Rank_Change",
        "AHP_Score",
        "AHP_Gaussian_Score",
    ]
]

,Project_ID,AHP_Rank,AHP_Gaussian_Rank,Rank_Change,Absolute_Rank_Change,AHP_Score,AHP_Gaussian_Score
2352,PROJ_1538,2353,1867,486,486,0.628908,0.660063
2577,PROJ_0668,2578,2126,452,452,0.612561,0.641909
2150,PROJ_3295,2151,1736,415,415,0.645344,0.669841
2089,PROJ_2247,2090,1675,415,415,0.649628,0.674390
2620,PROJ_2498,2621,2268,353,353,0.609571,0.629841
2349,PROJ_3989,2350,2008,342,342,0.629180,0.649899
2078,PROJ_0336,2079,1745,334,334,0.650402,0.669339
2585,PROJ_0071,2586,2278,308,308,0.612172,0.629151
1958,PROJ_1592,1959,1658,301,301,0.659088,0.675904
2266,PROJ_1629,2267,1976,291,291,0.636123,0.652333


In [43]:
rank_improvements = ahp_comparison[
    ahp_comparison["Rank_Change"] > 0
].sort_values(
    by="Rank_Change",
    ascending=False
)

rank_declines = ahp_comparison[
    ahp_comparison["Rank_Change"] < 0
].sort_values(
    by="Rank_Change"
)

print(
    "Projects that improved their ranking:",
    len(rank_improvements)
)

print(
    "Projects that declined in ranking:",
    len(rank_declines)
)

Projects that improved their ranking: 1827
Projects that declined in ranking: 2144


In [44]:
unchanged_projects = (
    ahp_comparison["Rank_Change"] == 0
).sum()

print(
    "Projects with unchanged ranking:",
    unchanged_projects
)

print(
    f"Percentage unchanged: "
    f"{unchanged_projects / len(ahp_comparison) * 100:.2f}%"
)

Projects with unchanged ranking: 29
Percentage unchanged: 0.73%


## 17. Compare AHP-Gaussian and Monte Carlo Rankings

The AHP-Gaussian ranking is compared with the Monte Carlo ranking to evaluate the consistency between the deterministic and simulation-based approaches.

The comparison focuses on ranking correlation and the overlap among the highest-priority projects.

In [45]:
monte_carlo_ranking = pd.read_csv(
    "../data/processed/monte_carlo_ranking.csv"
)

monte_carlo_ranking.head()

,Project_ID,MC_Mean,MC_Std,MC_Min,MC_Max,MC_P05,MC_P50,MC_P95,MC_Rank
0,PROJ_0614,0.899162,0.067527,0.474232,1.0,0.767684,0.913329,0.984689,1
1,PROJ_1686,0.891604,0.072260,0.508686,1.0,0.754343,0.904913,0.985095,2
2,PROJ_3497,0.888904,0.068610,0.477808,1.0,0.756694,0.902654,0.976561,3
3,PROJ_2031,0.887733,0.069666,0.508840,1.0,0.753177,0.901902,0.977941,4
4,PROJ_0940,0.885036,0.067031,0.465636,1.0,0.754435,0.898646,0.971103,5


In [46]:
monte_carlo_ranking["MC_Rank"] = (
    monte_carlo_ranking["MC_Mean"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

In [47]:
gaussian_mc_comparison = ahp_gaussian_ranking[
    [
        "Project_ID",
        "AHP_Gaussian_Rank",
        "AHP_Gaussian_Score"
    ]
].merge(
    monte_carlo_ranking[
        [
            "Project_ID",
            "MC_Rank",
            "MC_Mean",
            "MC_Std",
            "MC_P05",
            "MC_P50",
            "MC_P95"
        ]
    ],
    on="Project_ID",
    how="inner"
)

gaussian_mc_comparison.head()

,Project_ID,AHP_Gaussian_Rank,AHP_Gaussian_Score,MC_Rank,MC_Mean,MC_Std,MC_P05,MC_P50,MC_P95
0,PROJ_0614,1,0.945826,1,0.899162,0.067527,0.767684,0.913329,0.984689
1,PROJ_3497,2,0.935679,3,0.888904,0.068610,0.756694,0.902654,0.976561
2,PROJ_1686,3,0.932203,2,0.891604,0.072260,0.754343,0.904913,0.985095
3,PROJ_0940,4,0.930098,5,0.885036,0.067031,0.754435,0.898646,0.971103
4,PROJ_2031,5,0.926690,4,0.887733,0.069666,0.753177,0.901902,0.977941


In [48]:
gaussian_mc_comparison["Rank_Change"] = (
    gaussian_mc_comparison["AHP_Gaussian_Rank"]
    - gaussian_mc_comparison["MC_Rank"]
)

gaussian_mc_comparison["Absolute_Rank_Change"] = (
    gaussian_mc_comparison["Rank_Change"].abs()
)

## 18. Compare Ranking Correlation

Spearman's rank correlation is used to measure the similarity between the AHP-Gaussian and Monte Carlo rankings.

A higher correlation indicates that projects tend to maintain similar relative positions across the two approaches.

In [49]:
spearman_gaussian_mc, p_value_gaussian_mc = spearmanr(
    gaussian_mc_comparison["AHP_Gaussian_Rank"],
    gaussian_mc_comparison["MC_Rank"]
)

print(
    f"Spearman rank correlation: "
    f"{spearman_gaussian_mc:.4f}"
)

print(
    f"P-value: "
    f"{p_value_gaussian_mc:.4e}"
)

Spearman rank correlation: 0.9933
P-value: 0.0000e+00


In [50]:
score_correlation_gaussian_mc = (
    gaussian_mc_comparison[
        [
            "AHP_Gaussian_Score",
            "MC_Mean"
        ]
    ]
    .corr()
    .iloc[0, 1]
)

print(
    f"Score correlation: "
    f"{score_correlation_gaussian_mc:.4f}"
)

Score correlation: 0.9932


In [51]:
print(
    "Projects compared:",
    len(gaussian_mc_comparison)
)

Projects compared: 4000


## 19. Compare the Top 20 Projects

The Top 20 projects from the AHP-Gaussian and Monte Carlo rankings are compared to evaluate the consistency of the highest-priority projects identified by the two approaches.

In [52]:
top_20_gaussian = set(
    gaussian_mc_comparison
    .nsmallest(20, "AHP_Gaussian_Rank")["Project_ID"]
)

top_20_mc = set(
    gaussian_mc_comparison
    .nsmallest(20, "MC_Rank")["Project_ID"]
)

top_20_gaussian_mc_overlap = (
    top_20_gaussian & top_20_mc
)

print(
    f"Top-20 overlap: "
    f"{len(top_20_gaussian_mc_overlap)}/20 "
    f"({len(top_20_gaussian_mc_overlap) / 20 * 100:.1f}%)"
)

Top-20 overlap: 16/20 (80.0%)


In [53]:
top_20_only_gaussian = sorted(
    top_20_gaussian - top_20_mc
)

top_20_only_mc = sorted(
    top_20_mc - top_20_gaussian
)

print("Projects only in AHP-Gaussian Top 20:")
print(top_20_only_gaussian)

print("\nProjects only in Monte Carlo Top 20:")
print(top_20_only_mc)

Projects only in AHP-Gaussian Top 20:
['PROJ_0692', 'PROJ_0926', 'PROJ_1845', 'PROJ_2838']

Projects only in Monte Carlo Top 20:
['PROJ_1114', 'PROJ_1856', 'PROJ_2688', 'PROJ_3020']


In [54]:
top_20_gaussian_mc_comparison = (
    gaussian_mc_comparison[
        gaussian_mc_comparison["Project_ID"].isin(
            top_20_gaussian | top_20_mc
        )
    ]
    .sort_values(by="AHP_Gaussian_Rank")
)

top_20_gaussian_mc_comparison[
    [
        "Project_ID",
        "AHP_Gaussian_Rank",
        "MC_Rank",
        "Rank_Change",
        "AHP_Gaussian_Score",
        "MC_Mean",
        "MC_Std",
    ]
]

,Project_ID,AHP_Gaussian_Rank,MC_Rank,Rank_Change,AHP_Gaussian_Score,MC_Mean,MC_Std
0,PROJ_0614,1,1,0,0.945826,0.899162,0.067527
1,PROJ_3497,2,3,-1,0.935679,0.888904,0.068610
2,PROJ_1686,3,2,1,0.932203,0.891604,0.072260
3,PROJ_0940,4,5,-1,0.930098,0.885036,0.067031
4,PROJ_2031,5,4,1,0.926690,0.887733,0.069666
5,PROJ_2119,6,11,-5,0.921286,0.865159,0.073521
6,PROJ_3982,7,7,0,0.918222,0.869566,0.071762
7,PROJ_2845,8,6,2,0.914946,0.873105,0.073066
8,PROJ_1070,9,8,1,0.911107,0.867142,0.073486
9,PROJ_2068,10,14,-4,0.910133,0.862881,0.062957


## 20. Analyze Monte Carlo Score Stability

The Monte Carlo results provide a distribution of simulated scores for each project.

The standard deviation and percentile range are used to assess the uncertainty associated with each project's simulated score.

In [55]:
gaussian_mc_comparison["MC_P05_P95_Range"] = (
    gaussian_mc_comparison["MC_P95"]
    - gaussian_mc_comparison["MC_P05"]
)

In [56]:
most_stable_projects = (
    gaussian_mc_comparison
    .sort_values(by="MC_Std")
    .head(10)
)

most_stable_projects[
    [
        "Project_ID",
        "AHP_Gaussian_Rank",
        "MC_Rank",
        "MC_Mean",
        "MC_Std",
        "MC_P05",
        "MC_P50",
        "MC_P95",
        "MC_P05_P95_Range",
    ]
]

,Project_ID,AHP_Gaussian_Rank,MC_Rank,MC_Mean,MC_Std,MC_P05,MC_P50,MC_P95,MC_P05_P95_Range
3372,PROJ_2348,3373,3411,0.545852,0.047060,0.467437,0.546511,0.622253,0.154816
3762,PROJ_3027,3763,3781,0.480880,0.047395,0.400650,0.482394,0.555913,0.155263
3098,PROJ_2235,3099,3022,0.582267,0.049560,0.500347,0.582575,0.663337,0.162990
2512,PROJ_3371,2513,2415,0.623194,0.050694,0.540152,0.623117,0.706313,0.166161
3486,PROJ_0145,3487,3574,0.524627,0.051331,0.436982,0.526443,0.606265,0.169283
3202,PROJ_1342,3203,3217,0.565216,0.053199,0.476536,0.565958,0.651873,0.175336
2649,PROJ_0941,2650,2548,0.614758,0.053997,0.525759,0.614920,0.703534,0.177775
3328,PROJ_0757,3329,3466,0.538393,0.054099,0.446207,0.540174,0.624300,0.178093
2900,PROJ_1509,2901,3064,0.578575,0.054972,0.484852,0.580695,0.665161,0.180309
1910,PROJ_2536,1911,1856,0.658421,0.055554,0.564474,0.659865,0.747249,0.182775


In [57]:
least_stable_projects = (
    gaussian_mc_comparison
    .sort_values(
        by="MC_Std",
        ascending=False
    )
    .head(10)
)

least_stable_projects[
    [
        "Project_ID",
        "AHP_Gaussian_Rank",
        "MC_Rank",
        "MC_Mean",
        "MC_Std",
        "MC_P05",
        "MC_P50",
        "MC_P95",
        "MC_P05_P95_Range",
    ]
]

,Project_ID,AHP_Gaussian_Rank,MC_Rank,MC_Mean,MC_Std,MC_P05,MC_P50,MC_P95,MC_P05_P95_Range
3699,PROJ_3588,3700,3703,0.498817,0.112467,0.317098,0.496874,0.686366,0.369268
3153,PROJ_2797,3154,3222,0.564595,0.111961,0.383322,0.562657,0.751869,0.368547
3848,PROJ_1116,3849,3831,0.466190,0.111693,0.288388,0.462700,0.656003,0.367614
3733,PROJ_3003,3734,3678,0.503942,0.111029,0.331429,0.497845,0.695485,0.364056
2978,PROJ_0682,2979,3184,0.568554,0.110939,0.385165,0.568356,0.750997,0.365832
3608,PROJ_2984,3609,3682,0.503508,0.110679,0.325989,0.500984,0.689892,0.363903
2495,PROJ_3317,2496,2807,0.597932,0.110368,0.417059,0.597746,0.781114,0.364055
2795,PROJ_2314,2796,3061,0.578754,0.110196,0.394639,0.579590,0.758301,0.363662
3211,PROJ_0355,3212,3245,0.562354,0.110089,0.381869,0.561445,0.743515,0.361646
2910,PROJ_0457,2911,3156,0.570874,0.109961,0.387977,0.571964,0.750943,0.362966


In [58]:
top_20_gaussian_stability = (
    gaussian_mc_comparison[
        gaussian_mc_comparison["Project_ID"].isin(
            top_20_gaussian
        )
    ]
    .sort_values(by="AHP_Gaussian_Rank")
)

top_20_gaussian_stability[
    [
        "Project_ID",
        "AHP_Gaussian_Rank",
        "MC_Rank",
        "MC_Mean",
        "MC_Std",
        "MC_P05",
        "MC_P95",
    ]
]

,Project_ID,AHP_Gaussian_Rank,MC_Rank,MC_Mean,MC_Std,MC_P05,MC_P95
0,PROJ_0614,1,1,0.899162,0.067527,0.767684,0.984689
1,PROJ_3497,2,3,0.888904,0.068610,0.756694,0.976561
2,PROJ_1686,3,2,0.891604,0.072260,0.754343,0.985095
3,PROJ_0940,4,5,0.885036,0.067031,0.754435,0.971103
4,PROJ_2031,5,4,0.887733,0.069666,0.753177,0.977941
5,PROJ_2119,6,11,0.865159,0.073521,0.725787,0.964705
6,PROJ_3982,7,7,0.869566,0.071762,0.733102,0.963797
7,PROJ_2845,8,6,0.873105,0.073066,0.735643,0.969763
8,PROJ_1070,9,8,0.867142,0.073486,0.728975,0.965137
9,PROJ_2068,10,14,0.862881,0.062957,0.736401,0.938650


## 21. Identify Robustly Prioritized Projects

Projects that perform well in both the AHP-Gaussian and Monte Carlo rankings are considered more robust candidates for prioritization.

The analysis identifies projects that remain highly ranked across both approaches.

In [59]:
robust_top_20 = gaussian_mc_comparison[
    gaussian_mc_comparison["Project_ID"].isin(
        top_20_gaussian & top_20_mc
    )
].copy()

robust_top_20 = robust_top_20.sort_values(
    by="AHP_Gaussian_Rank"
)

robust_top_20[
    [
        "Project_ID",
        "AHP_Gaussian_Rank",
        "MC_Rank",
        "AHP_Gaussian_Score",
        "MC_Mean",
        "MC_Std",
        "MC_P05",
        "MC_P95",
    ]
]

,Project_ID,AHP_Gaussian_Rank,MC_Rank,AHP_Gaussian_Score,MC_Mean,MC_Std,MC_P05,MC_P95
0,PROJ_0614,1,1,0.945826,0.899162,0.067527,0.767684,0.984689
1,PROJ_3497,2,3,0.935679,0.888904,0.068610,0.756694,0.976561
2,PROJ_1686,3,2,0.932203,0.891604,0.072260,0.754343,0.985095
3,PROJ_0940,4,5,0.930098,0.885036,0.067031,0.754435,0.971103
4,PROJ_2031,5,4,0.926690,0.887733,0.069666,0.753177,0.977941
5,PROJ_2119,6,11,0.921286,0.865159,0.073521,0.725787,0.964705
6,PROJ_3982,7,7,0.918222,0.869566,0.071762,0.733102,0.963797
7,PROJ_2845,8,6,0.914946,0.873105,0.073066,0.735643,0.969763
8,PROJ_1070,9,8,0.911107,0.867142,0.073486,0.728975,0.965137
9,PROJ_2068,10,14,0.910133,0.862881,0.062957,0.736401,0.938650


In [60]:
robust_top_20["Mean_Rank"] = (
    robust_top_20[
        [
            "AHP_Gaussian_Rank",
            "MC_Rank"
        ]
    ].mean(axis=1)
)

robust_top_20.sort_values(
    by="Mean_Rank"
)

,Project_ID,AHP_Gaussian_Rank,AHP_Gaussian_Score,MC_Rank,MC_Mean,MC_Std,MC_P05,MC_P50,MC_P95,Rank_Change,Absolute_Rank_Change,MC_P05_P95_Range,Mean_Rank
0,PROJ_0614,1,0.945826,1,0.899162,0.067527,0.767684,0.913329,0.984689,0,0,0.217005,1.0
1,PROJ_3497,2,0.935679,3,0.888904,0.068610,0.756694,0.902654,0.976561,-1,1,0.219867,2.5
2,PROJ_1686,3,0.932203,2,0.891604,0.072260,0.754343,0.904913,0.985095,1,1,0.230752,2.5
3,PROJ_0940,4,0.930098,5,0.885036,0.067031,0.754435,0.898646,0.971103,-1,1,0.216668,4.5
4,PROJ_2031,5,0.926690,4,0.887733,0.069666,0.753177,0.901902,0.977941,1,1,0.224764,4.5
6,PROJ_3982,7,0.918222,7,0.869566,0.071762,0.733102,0.882791,0.963797,0,0,0.230696,7.0
7,PROJ_2845,8,0.914946,6,0.873105,0.073066,0.735643,0.885651,0.969763,2,2,0.234120,7.0
5,PROJ_2119,6,0.921286,11,0.865159,0.073521,0.725787,0.877283,0.964705,-5,5,0.238918,8.5
8,PROJ_1070,9,0.911107,8,0.867142,0.073486,0.728975,0.879510,0.965137,1,1,0.236162,8.5
11,PROJ_2435,12,0.900125,10,0.865201,0.072576,0.728730,0.877960,0.961228,2,2,0.232498,11.0


In [61]:
robust_top_5 = robust_top_20.sort_values(
    by="Mean_Rank"
).head(5)

robust_top_5[
    [
        "Project_ID",
        "AHP_Gaussian_Rank",
        "MC_Rank",
        "Mean_Rank",
        "MC_Mean",
        "MC_Std",
    ]
]

,Project_ID,AHP_Gaussian_Rank,MC_Rank,Mean_Rank,MC_Mean,MC_Std
0,PROJ_0614,1,1,1.0,0.899162,0.067527
1,PROJ_3497,2,3,2.5,0.888904,0.068610
2,PROJ_1686,3,2,2.5,0.891604,0.072260
3,PROJ_0940,4,5,4.5,0.885036,0.067031
4,PROJ_2031,5,4,4.5,0.887733,0.069666


## 22. Save AHP-Gaussian Results

The AHP-Gaussian ranking and the criterion weight analysis are saved as processed datasets for further analysis and reproducibility.


In [62]:
ahp_gaussian_ranking.to_csv(
    "../data/processed/ahp_gaussian_ranking.csv",
    index=False
)

weights_comparison.to_csv(
    "../data/processed/ahp_gaussian_weights.csv"
)

gaussian_mc_comparison.to_csv(
    "../data/processed/ahp_gaussian_monte_carlo_comparison.csv",
    index=False
)

print("AHP-Gaussian results saved successfully.")

AHP-Gaussian results saved successfully.


## 23. Validate Saved Results

The saved datasets are loaded again to verify that the main outputs were successfully generated and contain the expected number of projects.

In [63]:
saved_ranking = pd.read_csv(
    "../data/processed/ahp_gaussian_ranking.csv"
)

saved_weights = pd.read_csv(
    "../data/processed/ahp_gaussian_weights.csv",
    index_col=0
)

saved_comparison = pd.read_csv(
    "../data/processed/ahp_gaussian_monte_carlo_comparison.csv"
)

print(
    "AHP-Gaussian ranking:",
    saved_ranking.shape
)

print(
    "AHP-Gaussian weights:",
    saved_weights.shape
)

print(
    "AHP-Gaussian and Monte Carlo comparison:",
    saved_comparison.shape
)

AHP-Gaussian ranking: (4000, 3)
AHP-Gaussian weights: (6, 5)
AHP-Gaussian and Monte Carlo comparison: (4000, 12)


## 24. Summary of Results

The AHP-Gaussian method was applied to the complete portfolio of 4,000 projects.

The Gaussian adjustment incorporated the relative variability of the evaluation criteria into the conventional AHP weights. This resulted in changes in the relative importance of the criteria and, consequently, in the project ranking.

The comparison between AHP and AHP-Gaussian was used to identify changes in project prioritization. The AHP-Gaussian ranking was also compared with the Monte Carlo results to evaluate the consistency between the deterministic and simulation-based approaches.

The Monte Carlo analysis provides an additional perspective by quantifying the uncertainty associated with project scores, while the AHP-Gaussian approach incorporates criterion variability directly into the weighting process.

Together, these methods provide complementary information for project portfolio prioritization and can support decision-making under uncertainty.

## 25. Conclusion

The analysis demonstrates how combining AHP-Gaussian weighting with Monte Carlo simulation can provide a broader view of project portfolio prioritization.

AHP provides the initial structure for defining the relative importance of the criteria. The Gaussian adjustment incorporates the variability observed in the portfolio, while Monte Carlo simulation evaluates the uncertainty associated with project scores.

The results indicate that the methods produce broadly consistent prioritization patterns while also identifying projects whose relative positions are more sensitive to the treatment of criterion variability and uncertainty.

Therefore, the proposed approach should be considered a decision-support framework rather than a replacement for managerial judgment. Additional business constraints, strategic priorities, resource limitations, and project dependencies should be considered before final portfolio decisions are made.